# Mi validación de agency antes de publicar Silver

Ya revisé completitud y duplicados. Ahora aplicaré el contrato v1: formatos, conflictos y conservación de cada snapshot. Este notebook solo lee Bronze y Staging y muestra resultados en memoria. No escribe tablas, archivos de datos, auditorías ni cuarentena persistida.

Ejecutaré un bloque a la vez. `candidate_rows` significa registros que pasan las reglas; solo si no hay rechazados podremos preparar su publicación. `ready_for_publication=true` no escribe ni certifica una carga Delta.


## Bloque 0 — Preparar los validadores

Instalo las mismas versiones de `databricks/requirements-silver.txt`. Esto modifica las librerías de mi sesión, no las tablas. Si Databricks solicita reiniciar Python, lo haré antes de ejecutar desde el bloque 1. No volveré a ejecutar este bloque en medio del recorrido.


In [ ]:
%pip install langcodes==3.5.1 tzdata==2026.6


## Bloque 1 — Elegir el snapshot y cargar el código

Los defaults corresponden a mi entorno de aprendizaje y a 2025. Cambiaré versión y hash juntos para 2026. Necesito el repositorio completo con `databricks/gtfs_silver/`, no solo este notebook importado.

En un Git folder puede encontrarse el proyecto automáticamente. En serverless el directorio actual no está garantizado: si no se encuentra, configuraré `repo_root` con la ruta absoluta del repositorio en Workspace y repetiré este bloque. No pondré una ruta personal en el archivo versionado.


In [ ]:
# BLOQUE 1 — Parámetros, código versionado y zona de sesión
from pathlib import Path
import re
import sys

dbutils.widgets.text("repo_root", "", "Ruta absoluta del repositorio, si hace falta")
dbutils.widgets.text("catalog", "workspace", "Catálogo")
dbutils.widgets.text("schema", "gtfs_bogota", "Schema")
dbutils.widgets.text("staging_volume", "staging_gtfs", "Volume Staging")
dbutils.widgets.text("feed_version", "2025-10-28", "Versión GTFS")
dbutils.widgets.text("snapshot_sha256", "07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952", "SHA-256")

configured_root = dbutils.widgets.get("repo_root").strip()
if configured_root:
    if not Path(configured_root).is_absolute():
        raise ValueError("repo_root debe ser una ruta absoluta de Workspace.")
    possible_roots = [Path(configured_root).resolve()]
else:
    current_directory = Path.cwd().resolve()
    possible_roots = [current_directory, *current_directory.parents]
project_root = next(
    (root for root in possible_roots if (root / "databricks/gtfs_silver/agency_validation.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("No encuentro gtfs_silver. Sincroniza el repositorio completo y configura repo_root.")
module_directory = str(project_root / "databricks")
if module_directory not in sys.path:
    sys.path.insert(0, module_directory)

from gtfs_silver.agency_validation import SnapshotSpec, read_source_header, reference_versions
from gtfs_silver.agency_spark import read_and_validate, result_frames

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
staging_volume = dbutils.widgets.get("staging_volume").strip()
for parameter_name, parameter_value in (("catalog", catalog), ("schema", schema), ("staging_volume", staging_volume)):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", parameter_value):
        raise ValueError(f"{parameter_name}: usa un identificador simple.")
feed_version = dbutils.widgets.get("feed_version").strip()
snapshot_sha256 = dbutils.widgets.get("snapshot_sha256").strip().lower()
agency_source_path = f"/Volumes/{catalog}/{schema}/{staging_volume}/gtfs_static/feed_version={feed_version}/sha256={snapshot_sha256}/agency.txt"
snapshot_spec = SnapshotSpec(feed_version, snapshot_sha256, agency_source_path)
agency_table = f"`{catalog}`.`{schema}`.`bronze_gtfs_agency`"
spark.conf.set("spark.sql.session.timeZone", "UTC")
print(f"Snapshot: {feed_version}; SHA-256: {snapshot_sha256}")
print(f"Referencias de validación: {reference_versions()}")
print("Ejecución de lectura. Límite de seguridad: 1.000 filas de agency.")


## Bloque 2 — Leer el encabezado publicado

El encabezado manda sobre las columnas históricas. Si en 2025 no se publicó `agency_fare_url`, su contenido no se tomará de otra versión. Si falta una columna requerida o aparece una no contemplada, revisaré el contrato antes de continuar.


In [ ]:
# BLOQUE 2 — Encabezado fuente y existencia de Bronze
published_columns = read_source_header(agency_source_path)
if not spark.catalog.tableExists(agency_table):
    raise RuntimeError("No existe la tabla Bronze de agency; comprueba catálogo y schema.")
agency_bronze_df = spark.table(agency_table)
print(f"Columnas publicadas: {list(published_columns)}")


## Bloque 3 — Aplicar las reglas y reconciliar

Los validadores de URL, idioma y zona trabajan sobre una referencia pequeña de agency. Spark filtra el hash y lee como máximo 1.001 filas para detectar un exceso del límite de 1.000; después se validan en Python sin UDF y se devuelven DataFrames con esquema explícito. Esta estrategia no se usará para trips o stop_times.

Leer por hash antes de comprobar la versión permite detectar metadata contradictoria. Las copias válidas idénticas se consolidan solo en memoria. Los conflictos de ID rechazan todas sus filas. Un snapshot con rechazos queda bloqueado aunque conserve candidatos válidos.

La suma debe ser: entrada = candidatos + rechazados + copias consolidadas. Si reinicié Python, repetiré desde el bloque 1.


In [ ]:
# BLOQUE 3 — Evaluación determinista sin persistencia
agency_validation = read_and_validate(
    agency_bronze_df, published_columns, snapshot_spec, max_rows=1000,
)
agency_frames = result_frames(spark, agency_validation)
print("Resumen de la validación:")
for metric, value in agency_validation.summary.items():
    print(f"  {metric}: {value}")
if not agency_validation.summary["reconciled"]:
    raise RuntimeError("La reconciliación no cuadra; no continuar.")
print("APTO PARA REVISAR PUBLICACIÓN" if agency_validation.summary["ready_for_publication"] else "BLOQUEADO: revisar los motivos del bloque 4")
print("No se ha escrito ninguna tabla Silver.")


## Bloque 4 — Entender candidatos y motivos

Los candidatos tienen los ocho campos de negocio como string y la procedencia. Los opcionales sin contenido aparecen como null. Una tabla de rechazados o advertencias vacía significa que ese control no encontró casos.

`error_codes` explica por qué se rechazó una fila y `original_row_json` conserva lo leído de Bronze. No es cuarentena persistida. Las advertencias no bloqueantes conservan el valor original, por ejemplo espacios extremos en un nombre. Un ID con espacios sí bloquea, porque recortarlo podría cambiar una relación.


In [ ]:
# BLOQUE 4 — Muestras acotadas para revisar antes de transformar
from gtfs_silver.agency_validation import BUSINESS_FIELDS

print("Candidatos válidos en memoria:")
display(agency_frames["candidates"].select(*BUSINESS_FIELDS).orderBy("agency_id").limit(20))
print("Rechazados y motivos:")
display(agency_frames["rejected"].limit(20))
print("Advertencias que conservan el valor publicado:")
display(agency_frames["warnings"].limit(20))
print("Fin de la validación de lectura. La escritura Delta y sus pruebas se prepararán después.")


## Mi comprobación antes de continuar

Primero revisaré 2025. Si no aparecen rechazos, espero siete candidatos y cero copias consolidadas según el perfilado anterior; no impongo ese conteo a versiones futuras. Después ejecutaré 1 → 2 → 3 → 4 con 2026 y su hash documentado.

La clave sigue siendo versión + hash + agency_id. Todavía no existen silver_run_id ni silver_loaded_at_utc en los candidatos porque no hubo carga Silver. Tampoco he probado reemplazo de snapshots ni reejecución Delta.

Referencias: contrato en `docs/contracts/agency.md`, runbook en `docs/notebook_runbook.md`, [módulos en Workspace](https://docs.databricks.com/aws/en/files/workspace-modules). Este notebook está preparado para ejecutar; sus resultados no están guardados en el repositorio.
